# 04 — Project-page teaser figure

Builds the hero figure for the project webpage: a side-by-side panel showing
(left) max disparity across slices on each metric, (right) the worst-slice
stereotype rate by prompt family with the 5% audit threshold.

Output: `figures/teaser.pdf` and `figures/teaser.png`.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

ROOT = Path('..').resolve()
DATA = ROOT / 'analysis/big_study_scored'
FIG = ROOT / 'figures'
FIG.mkdir(exist_ok=True)

plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})


In [ ]:
disp = pd.read_csv(DATA / 'summary_disparities.csv').sort_values('max_minus_min')
fam = pd.read_csv(DATA / 'summary_by_slice_prompt.csv')

label_map = {
    'refusal_rate': 'Refusal',
    'deflection_rate': 'Deflection',
    'demeaning_kw_rate': 'Demeaning kw.',
    'identity_salience_rate': 'Identity salience',
    'religion_mention_rate': 'Religion mention',
    'pilot_harm_rate': 'Pilot harm',
    'stereotype_kw_rate': 'Stereotype kw.',
    'title_mention_rate': 'Title mention',
}
families = ['F1', 'F2', 'F3', 'F4']
slices = ['man_Christian', 'man_Muslim', 'woman_Christian', 'woman_Muslim']
slice_labels = ['M × Christian', 'M × Muslim', 'F × Christian', 'F × Muslim']
# Paul Tol "vibrant" palette: colorblind-safe.
colors = ['#0077BB', '#EE7733', '#009988', '#EE3377']

fig, (axL, axR) = plt.subplots(1, 2, figsize=(10, 3.5), gridspec_kw={'width_ratios': [1, 1.1]})

# Left: max disparities
axL.barh([label_map[m] for m in disp.metric], disp.max_minus_min, color='#0077BB')
axL.axvline(0.05, color='black', linestyle=':', linewidth=1)
from matplotlib.lines import Line2D
threshold_handle = Line2D([], [], color='black', linestyle=':', linewidth=1,
                          label='5% audit threshold')
xmid = disp.max_minus_min.max() / 2
axL.legend(handles=[threshold_handle], loc='center',
           bbox_to_anchor=(xmid, 0), bbox_transform=axL.transData,
           frameon=False, fontsize=7.5)
axL.set_xlabel('Max disparity across demographic slices')
axL.set_title('Slice disparities look small', loc='left', fontweight='bold')

# Right: worst-slice stereotype rate per family
pivot = fam.pivot(index='prompt_family', columns='full_slice_id', values='stereotype_kw_rate')
pivot = pivot.loc[families, slices]
x = np.arange(len(families))
w = 0.2
for i, (sl, lbl) in enumerate(zip(slices, slice_labels)):
    axR.bar(x + (i - 1.5) * w, pivot[sl], width=w, label=lbl, color=colors[i])
axR.axhline(0.05, color='black', linestyle=':', linewidth=1, label='5% audit threshold')
axR.set_xticks(x)
axR.set_xticklabels(families)
axR.set_xlabel('Prompt family')
axR.set_ylabel('Stereotype-keyword rate')
axR.set_title('… until you change the prompt family', loc='left', fontweight='bold')
axR.legend(loc='upper right', frameon=False, fontsize=7.5)

fig.tight_layout()
fig.savefig(FIG / 'teaser.pdf')
fig.savefig(FIG / 'teaser.png', dpi=200)
plt.show()
